In [1]:
!pip install fastapi uvicorn nest_asyncio
!pip install transformers sentencepiece accelerate
!pip install pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 110.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 84.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 50.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 108.7 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjit

In [3]:
from pyngrok import ngrok
from huggingface_hub import login

#ngrok 토큰
ngrok.set_auth_token("")
#huggingface 토큰
login("")

In [ ]:
from fastapi import FastAPI, Request
from pydantic import BaseModel
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM
from pyngrok import ngrok
import nest_asyncio
import uvicorn
import torch



# model_name = "Bllossom/llama-3.2-Korean-Bllossom-AICA-5B"
model_name = "MLP-KTLim/llama-3-Korean-Bllossom-8B"


tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)
model.eval()



# if torch.cuda.is_available():
#     model.to("cuda")

# 요청 형식 정의
# class QueryRequest(BaseModel):
#     prompt: str



# FastAPI 앱 초기화
app = FastAPI()


# POST 요청 처리
@app.post("/inference")
async def generate_answer(request: Request):

    body = await request.json()
    prompt = body.get("prompt", "")

    inputs = tokenizer(prompt, return_tensors="pt", padding=True)
    input_ids = inputs["input_ids"].to(model.device)
    attention_mask = inputs["attention_mask"].to(model.device)

    with torch.no_grad():
        output = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=512,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id  # 경고 방지용 설정
        )

    # 전체 디코딩 결과
    decoded = tokenizer.decode(output[0], skip_special_tokens=True)

    #  prompt 이후 생성된 부분만 추출
    if decoded.startswith(prompt):
        answer_text = decoded[len(prompt):].strip()
    else:
        answer_text = decoded

    return {"result": answer_text}


@app.get("/")
async def root():
    return {"message": "FastAPI 서버가 실행 중입니다."}


# Colab 환경 대응 (ngrok + asyncio 우회)
public_url = ngrok.connect(7860)
print("외부 접속 주소:", public_url)

# Colab 환경에서는 asyncio 충돌 우회 필요
nest_asyncio.apply()

# uvicorn 대신 내부에서 직접 서버 실행
uvicorn.run(app, host="0.0.0.0", port=7860)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/710 [00:00<?, ?B/s]

...

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/172 [00:00<?, ?B/s]

외부 접속 주소: NgrokTunnel: "https://bea141a5b643.ngrok-free.app" -> "http://localhost:7860"


INFO:     Started server process [1155]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:7860 (Press CTRL+C to quit)


INFO:     101.235.186.162:0 - "GET / HTTP/1.1" 200 OK
INFO:     101.235.186.162:0 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     101.235.186.162:0 - "POST /inference HTTP/1.1" 200 OK
INFO:     101.235.186.162:0 - "POST /inference HTTP/1.1" 200 OK
INFO:     101.235.186.162:0 - "POST /inference HTTP/1.1" 200 OK
INFO:     101.235.186.162:0 - "POST /inference HTTP/1.1" 200 OK
INFO:     101.235.186.162:0 - "POST /inference HTTP/1.1" 200 OK
INFO:     101.235.186.162:0 - "POST /inference HTTP/1.1" 200 OK
INFO:     101.235.186.162:0 - "POST /inference HTTP/1.1" 200 OK
INFO:     101.235.186.162:0 - "POST /inference HTTP/1.1" 200 OK
INFO:     101.235.186.162:0 - "POST /inference HTTP/1.1" 200 OK
INFO:     101.235.186.162:0 - "POST /inference HTTP/1.1" 200 OK
INFO:     101.235.186.162:0 - "POST /inference HTTP/1.1" 200 OK
INFO:     101.235.186.162:0 - "POST /inference HTTP/1.1" 200 OK
INFO:     101.235.186.162:0 - "POST /inference HTTP/1.1" 200 OK
INFO:     101.235.186.162:0 - "POST /infer